!# Constant-impedance CPW taper

Transitions from `(W1, S1)` to `(W2, S2)`. `W` follows a chosen profile, and the gap `S` is solved at each section to hold the impedance.

**Limitation: the taper length is your input, not a result.** Both skrf and uSimmics use a circuit model that only sees impedance steps. It has no step parasitics, mode conversion or radiation, so a constant-impedance line shows almost no reflection at any length. The `S11` plots below check the schematic and the impedance profile, not whether the taper is long enough. Use the edge angles as a guide and a full-wave simulation to validate the length.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import skrf as rf
from pathlib import Path
from skrf.mathFunctions import mag_2_db

from skrf_usimmics import Substrate, constant_z_cpw, usimmics
from skrf_usimmics.plots import plot_outline, plot_profile, plot_s11

In [ ]:
# Geometry and substrate (all lengths in m)
W1 = 40e-6
S1 = 25e-6
W2 = 10e-6
H = 500e-6
EP_R = 11.7

LENGTH = 500e-6
N_SECTIONS = 20
F_DESIGN = 10e9         # frequency at which the impedance is evaluated

freq = rf.Frequency(0.1, 30, 401, 'GHz')
substrate = Substrate(h=H, ep_r=EP_R)
PROJECT = Path.home() / '.qucs' / 'CPW_const_z_prj'


def study(w2, s2, sch_name, s11_ylim):
    line = constant_z_cpw(W1, S1, w2, s2, LENGTH, N_SECTIONS, substrate, freq, F_DESIGN)
    report = line.report
    print(f'Z0: {report["z1"]:.2f} -> {report["z2"]:.2f} ohm (endpoint mismatch {report["endpoint_mismatch_ohm"]:+.2f} ohm)')
    print(f'S2 for constant impedance at W2 = {w2*1e6:g} um: {report["s2_for_constant_z"]*1e6:.3f} um (using {s2*1e6:.3f} um)')
    print(f'Largest error from the target impedance: {report["z_error_ohm"]:.2e} ohm')
    print(f'Minimum gap: {report["min_gap"]*1e6:.3f} um')
    print(f'Steepest edge angle: inner {report["inner_edge_angle_deg"]:.2f} deg, outer {report["outer_edge_angle_deg"]:.2f} deg')

    plot_profile(line)
    plot_outline(line)
    plt.show()
    display(line.table().round(3))

    network = line.network(freq)
    network.name = 'skrf cascade'
    path = PROJECT / sch_name
    status = usimmics.write_schematic(path, line, freq)
    print(f'Schematic {status}: {path}')
    from_usimmics = usimmics.read_sp(usimmics.run_simulation(path), list(line.z_ports))

    plot_s11([network, from_usimmics], 'skrf and uSimmics reflection', ylim=s11_ylim)
    plt.show()
    print(f'Peak |S11|: {mag_2_db(np.abs(network.s[:, 0, 0])).max():.2f} dB')
    print(f'Max |S11 difference|, skrf vs uSimmics: {np.abs(network.s[:, 0, 0] - from_usimmics.s[:, 0, 0]).max():.2e}')
    return line

!## Case 1: matched endpoints

`S2` is chosen to give the same impedance as `(W1, S1)`, so the impedance stays constant along the taper.

In [ ]:
Z1 = substrate.z0(freq, F_DESIGN, W1, S1)
S2_MATCHED = substrate.gap_for_z0(freq, F_DESIGN, W2, Z1)

line_matched = study(W2, S2_MATCHED, 'const_z.sch', s11_ylim=(-120, 0))

!## Case 2: mismatched endpoints

`(W2, S2)` is fixed at 10 um / 10 um, which has a different impedance. Both endpoints are still reached exactly, so the impedance ramps between them.

Here 0.5 mm is only about 45 degrees long at 30 GHz, so the ramp behaves almost like an abrupt step between the two impedances, and `S11` stays near -23 dB across the band. If you need a real impedance change, use `klopfenstein_cpw`, or make the ramp much longer.

In [ ]:
line_ramp = study(W2, 10e-6, 'z_ramp.sch', s11_ylim=(-60, 0))